# 08 Cited Documents Extraction 

In [1]:
import pandas as pd, re
from collections import Counter


CSV      = "cleaned_comments_ver2.csv"   
ID_COL   = "Document ID"
TEXT_COL = "comment_text"                

df  = pd.read_csv(CSV, low_memory=False)
txt = df[TEXT_COL].fillna("").astype(str)
print("rows:", len(df))

rows: 10195


In [2]:

WHITELIST_ACRO = {
 "NEPA","DWPA","EIS","DEIS","SDEIS","FEIS","ESA","CWA","CAA","CZMA","NHPA",
 "ROD","MARAD","USACE","USCG","EPA","NOAA","USFWS","MMPA","APA","OCSLA","CFR","MBTA"
}

ACT_RE  = re.compile(r"\b([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){0,3}\s+Act)\b")   # X Act
ACRO_RE = re.compile(r"\b([A-Z]{2,6})\b")                                          # NEPA, DWPA...
SEC_RE  = re.compile(r"\b(Section\s+\d+[A-Za-z]?)\b")                             # Section 404
CFR_RE  = re.compile(r"\b(\d+\s+CFR\s+\d+)\b")                                  # 33 CFR 148
EO_RE   = re.compile(r"\b(Executive Order\s+\d+)\b")                             # Executive Order 14008
EIS_RE  = re.compile(r"\b((?:Draft|Final|Supplemental)\s+Environmental Impact Statement)\b")

def extract(t):
    found=set()
    for m in ACT_RE.findall(t):  found.add(m.strip())
    for m in ACRO_RE.findall(t):
        if m in WHITELIST_ACRO:  found.add(m)
    for m in SEC_RE.findall(t):  found.add(m)
    for m in CFR_RE.findall(t):  found.add(m)
    for m in EO_RE.findall(t):   found.add(m)
    for m in EIS_RE.findall(t):  found.add(m)
    return sorted(found)

In [3]:

df["cited_documents"] = txt.apply(extract)
df["n_cited"]         = df["cited_documents"].apply(len)


out = df[[ID_COL, "cited_documents", "n_cited"]].copy()
out["cited_documents"] = out["cited_documents"].apply(lambda l: "|".join(l))
out.to_csv("cited_documents_per_comment.csv", index=False)
print("Wrote cited_documents_per_comment.csv")
print("comments citing >=1 doc:", int((df['n_cited']>0).sum()), "/", len(df))
print("mean citations per comment:", round(df['n_cited'].mean(),2))

Wrote cited_documents_per_comment.csv
comments citing >=1 doc: 6866 / 10195
mean citations per comment: 3.03


In [4]:

freq = Counter()
for lst in df["cited_documents"]:
    for d in lst: freq[d]+=1

freq_df = (pd.DataFrame(freq.most_common(), columns=["document","n_comments"])
           .assign(pct=lambda x: (100*x["n_comments"]/len(df)).round(1)))
freq_df.to_csv("cited_documents_frequency.csv", index=False)
freq_df.head(25)

,document,n_comments,pct
0,MARAD,6734,66.1
1,NEPA,4101,40.2
2,Deepwater Water Port Act,4086,40.1
3,DWPA,4082,40.0
4,FEIS,3789,37.2
5,EPA,2554,25.1
6,EIS,2440,23.9
7,DEIS,2422,23.8
8,National Environmental Policy Act,278,2.7
9,NOAA,101,1.0


## Caveat on interpretation 

In [5]:

sample = df[df["n_cited"]>0].sample(min(25, (df["n_cited"]>0).sum()), random_state=42)
for _,r in sample.iterrows():
    print(r[ID_COL], "->", r["cited_documents"])
    print("   ", str(r[TEXT_COL])[:200].replace("\n"," "))
    print()
print("Manually confirm how many are correct -> report a rough precision, e.g. 27/30 = 90%.")

MARAD-2019-0093-1120 -> ['DEIS', 'EIS', 'EPA', 'MARAD']
    I am writing to express opposition to Texas GulfLink LLC's application for a Deepwater Port License from the Maritime Administration, and raise concerns with significant inadequacies in the project's 

MARAD-2019-0093-10351 -> ['DWPA', 'Deepwater Water Port Act', 'FEIS', 'MARAD', 'NEPA']
    As an environmentally motivated voter and consumer, I voice my opposition to the Texas GulfLink oil export terminal. The final environmental impact statement released by MARAD fails to provide critica

MARAD-2019-0093-1511 -> ['DEIS', 'EIS', 'EPA', 'MARAD']
    I am writing to express opposition to Texas GulfLink LLC's application for a Deepwater Port License from the Maritime Administration, and raise concerns with significant inadequacies in the project's 

MARAD-2019-0093-2590 -> ['DEIS', 'EIS', 'EPA', 'MARAD']
    I am writing to express opposition to Texas GulfLink LLC's application for a Deepwater Port License from the Maritime Admin